# RAG Poisoning Detection — Phase 3 & 4

**Phase 3**: Train DeBERTa-v3-small classifier + XGBoost aggregator  
**Phase 4**: Score all 2400 triplets with 5 signals, apply filter, save outputs



In [ ]:
!pip install -q transformers==4.44.0 sentence-transformers xgboost scikit-learn pydantic pyyaml tqdm

In [ ]:
import sys
import os
import json
import pickle
import shutil
from pathlib import Path
import torch
from tqdm.auto import tqdm

# Point imports at the dataset — adjust if your dataset slug differs
DATASET_DIR = Path("/kaggle/input/thesis-phase34")
WORKING_DIR = Path("/kaggle/working")

sys.path.insert(0, str(DATASET_DIR))

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from src.utils.config import load_config
from src.dataset.splitter import load_splits

config = load_config(DATASET_DIR / "config" / "v2.yaml")

# Override all paths to write into /kaggle/working/
config["experiment"]["results_dir"] = str(WORKING_DIR / "results")
config["prefilter"]["classifier"]["checkpoint_dir"] = str(WORKING_DIR / "results" / "models" / "deberta_classifier")
config["prefilter"]["aggregation"]["xgboost"]["checkpoint_path"] = str(WORKING_DIR / "results" / "models" / "xgboost_aggregator.pkl")

splits = load_splits(DATASET_DIR / "data" / "splits")
train_triplets = splits["train"]
val_triplets   = splits["val"]
test_triplets  = splits["test"]

print(f"Loaded — train: {len(train_triplets)}, val: {len(val_triplets)}, test: {len(test_triplets)}")

## Phase 3a — Train DeBERTa-v3-small Classifier

In [ ]:
from src.prefilter.train_classifier import train_classifier

deberta_f1 = train_classifier(train_triplets, val_triplets, config)

In [ ]:
import copy

roberta_config = copy.deepcopy(config)
roberta_config["prefilter"]["classifier"]["model"] = "roberta-base"
roberta_config["prefilter"]["classifier"]["checkpoint_dir"] = str(WORKING_DIR / "results" / "models" / "roberta_classifier")

roberta_f1 = train_classifier(train_triplets, val_triplets, roberta_config)

# Save classifier comparison for ablation
clf_comparison = {"deberta_val_f1": deberta_f1, "roberta_val_f1": roberta_f1}
models_dir = WORKING_DIR / "results" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
(models_dir / "classifier_comparison.json").write_text(json.dumps(clf_comparison, indent=2))
print(f"\nDeBERTa val F1: {deberta_f1:.4f}  |  RoBERTa val F1: {roberta_f1:.4f}")

In [ ]:
from src.prefilter import embedding_signal, entropy_signal, classifier_signal, crossencoder_signal, answer_span_signal
from src.prefilter.aggregator import train_xgboost
from src.dataset.schema import PrefilterScore

all_val_scores = []
for t in tqdm(val_triplets, desc="Scoring val set for XGBoost"):
    emb = embedding_signal.score_triplet(t, config)
    ent = entropy_signal.score_triplet(t, config)
    clf = classifier_signal.score_triplet(t, config)
    ce  = crossencoder_signal.score_triplet(t, config)
    ans = answer_span_signal.score_triplet(t, config)
    for j in range(len(emb)):
        combined = PrefilterScore(
            triplet_id=t.id,
            passage_index=j,
            embedding_score=emb[j].embedding_score if j < len(emb) else 0.0,
            entropy_score=ent[j].entropy_score if j < len(ent) else 0.0,
            classifier_score=clf[j].classifier_score if j < len(clf) else 0.5,
            crossencoder_score=ce[j].crossencoder_score if j < len(ce) else 0.0,
            answer_span_score=ans[j].answer_span_score if j < len(ans) else 0.5,
            aggregated_score=0.0,
            flagged=False,
            ground_truth_poisoned=emb[j].ground_truth_poisoned if j < len(emb) else False,
        )
        all_val_scores.append(combined)

train_xgboost(all_val_scores, config)
print("Phase 3b complete.")

## Phase 3c — Choose Aggregation Method (Val Set Evaluation)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
from src.prefilter.aggregator import aggregate
import copy

# all_val_scores already computed above — reuse them
methods = ["weighted_vote", "xgboost", "majority_vote"]
results = {}

for method in methods:
    scores_copy = copy.deepcopy(all_val_scores)
    aggregate(scores_copy, config, method=method)
    y_true = [int(s.ground_truth_poisoned) for s in scores_copy]
    y_pred = [int(s.flagged) for s in scores_copy]
    results[method] = {
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
    }

print(f"{'Method':<20} {'F1':>6} {'Precision':>10} {'Recall':>8}")
print("-" * 48)
for method, m in results.items():
    print(f"{method:<20} {m['f1']:>6.3f} {m['precision']:>10.3f} {m['recall']:>8.3f}")

best_method = max(results, key=lambda m: results[m]["f1"])
print(f"\nBest method by F1: {best_method}  (note: xgboost is in-sample on val)")
print(f"config['prefilter']['aggregation']['primary'] is currently: '{config['prefilter']['aggregation']['primary']}'")
print("\nTo override before Phase 4, run:")
print("  config['prefilter']['aggregation']['primary'] = '<method>'")


In [ ]:
## Phase 3d — Evaluate prefilter on held-out TEST set
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
import copy

all_test_scores = []
for t in tqdm(test_triplets, desc="Scoring test set"):
    emb = embedding_signal.score_triplet(t, config)
    ent = entropy_signal.score_triplet(t, config)
    clf = classifier_signal.score_triplet(t, config)
    ce  = crossencoder_signal.score_triplet(t, config)
    ans = answer_span_signal.score_triplet(t, config)
    for j in range(len(emb)):
        ps = PrefilterScore(
            triplet_id=t.id, passage_index=j,
            embedding_score=emb[j].embedding_score if j < len(emb) else 0.0,
            entropy_score=ent[j].entropy_score if j < len(ent) else 0.0,
            classifier_score=clf[j].classifier_score if j < len(clf) else 0.5,
            crossencoder_score=ce[j].crossencoder_score if j < len(ce) else 0.0,
            answer_span_score=ans[j].answer_span_score if j < len(ans) else 0.5,
            aggregated_score=0.0, flagged=False,
            ground_truth_poisoned=emb[j].ground_truth_poisoned if j < len(emb) else False,
        )
        all_test_scores.append(ps)

# Evaluate each aggregation method on test set
print("
Test set prefilter evaluation (held-out):")
print(f"{'Method':<20} {'F1':<8} {'Precision':<12} {'Recall':<10} {'FPR (clean)':<12}")
print("-" * 65)
test_results = {}
for method in ["weighted_vote", "xgboost", "majority_vote"]:
    scores_copy = copy.deepcopy(all_test_scores)
    aggregate(scores_copy, config, method=method)
    y_true = [int(s.ground_truth_poisoned) for s in scores_copy]
    y_pred = [int(s.flagged) for s in scores_copy]
    f1  = f1_score(y_true, y_pred, zero_division=0)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    fpr = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1) / max(1, sum(1 for yt in y_true if yt == 0))
    test_results[method] = {"f1": f1, "precision": pre, "recall": rec, "fpr_clean": fpr}
    print(f"{method:<20} {f1:<8.3f} {pre:<12.3f} {rec:<10.3f} {fpr:<12.3f}")

# Save test evaluation results
test_eval_path = WORKING_DIR / "results" / "prefilter_scores" / "test_eval.json"
test_eval_path.parent.mkdir(parents=True, exist_ok=True)
test_eval_path.write_text(json.dumps(test_results, indent=2))
print(f"
Saved test evaluation to {test_eval_path}")


In [ ]:
from src.prefilter.aggregator import aggregate

all_triplets = train_triplets + val_triplets + test_triplets

all_emb, all_ent, all_clf, all_ce, all_ans, all_agg = [], [], [], [], [], []

for t in tqdm(all_triplets, desc="Scoring all triplets"):
    emb = embedding_signal.score_triplet(t, config)
    ent = entropy_signal.score_triplet(t, config)
    clf = classifier_signal.score_triplet(t, config)
    ce  = crossencoder_signal.score_triplet(t, config)
    ans = answer_span_signal.score_triplet(t, config)

    combined = []
    for j in range(len(emb)):
        ps = PrefilterScore(
            triplet_id=t.id,
            passage_index=j,
            embedding_score=emb[j].embedding_score if j < len(emb) else 0.0,
            entropy_score=ent[j].entropy_score if j < len(ent) else 0.0,
            classifier_score=clf[j].classifier_score if j < len(clf) else 0.5,
            crossencoder_score=ce[j].crossencoder_score if j < len(ce) else 0.0,
            answer_span_score=ans[j].answer_span_score if j < len(ans) else 0.5,
            aggregated_score=0.0,
            flagged=False,
            ground_truth_poisoned=emb[j].ground_truth_poisoned if j < len(emb) else False,
        )
        combined.append(ps)

    aggregate(combined, config)

    all_emb.extend([ps.model_copy(update={"aggregated_score": ps.embedding_score}) for ps in emb])
    all_ent.extend([ps.model_copy(update={"aggregated_score": ps.entropy_score}) for ps in ent])
    all_clf.extend([ps.model_copy(update={"aggregated_score": ps.classifier_score}) for ps in clf])
    all_ce.extend([ps.model_copy(update={"aggregated_score": ps.crossencoder_score}) for ps in ce])
    all_ans.extend([ps.model_copy(update={"aggregated_score": ps.answer_span_score}) for ps in ans])
    all_agg.extend(combined)

out_dir = WORKING_DIR / "results" / "prefilter_scores"
out_dir.mkdir(parents=True, exist_ok=True)

for name, scores in tqdm([
    ("embedding", all_emb), ("entropy", all_ent), ("classifier", all_clf),
    ("crossencoder", all_ce), ("answer_span", all_ans), ("aggregated", all_agg)
], desc="Saving scores"):
    path = out_dir / f"{name}_scores.json"
    path.write_text(json.dumps([s.model_dump() for s in scores], indent=2))
    tqdm.write(f"Saved {len(scores)} {name} scores")

print("Phase 4 scoring complete.")

In [ ]:
# Apply filter using already-computed aggregated scores
separator = config["dataset"]["context_separator"]

# Build per-triplet passage-index → flagged lookup
from collections import defaultdict
flagged_lookup = defaultdict(set)  # triplet_id -> set of flagged passage indices
for ps in all_agg:
    if ps.flagged:
        flagged_lookup[ps.triplet_id].add(ps.passage_index)

all_triplets_combined = train_triplets + val_triplets + test_triplets
filtered_triplets = []
n_modified = 0

for triplet in all_triplets_combined:
    ctx = triplet.poisoned_context
    passages = [p.strip() for p in ctx.split(separator) if p.strip()]
    flagged = flagged_lookup.get(triplet.id, set())
    kept = [p for i, p in enumerate(passages) if i not in flagged]
    new_ctx = separator.join(kept) if kept else ctx
    if new_ctx != ctx:
        n_modified += 1
    filtered_triplets.append(triplet.model_copy(update={"poisoned_context": new_ctx}))

filter_path = out_dir / "filtered_triplets.json"
filter_path.write_text(json.dumps([t.model_dump() for t in filtered_triplets], indent=2))
print(f"Filtered {len(filtered_triplets)} triplets — {n_modified} modified (passages removed)")
print(f"Saved to {filter_path}")


## Save outputs as zip for download

In [ ]:
shutil.make_archive(
    str(WORKING_DIR / "phase34_results"),
    "zip",
    str(WORKING_DIR / "results"),
)
print("Zipped to /kaggle/working/phase34_results.zip")
print("\nContents:")
for p in sorted((WORKING_DIR / "results").rglob("*")):
    if p.is_file():
        size = p.stat().st_size / 1024
        print(f"  {p.relative_to(WORKING_DIR / 'results')}  ({size:.0f} KB)")